<a href="https://colab.research.google.com/github/YashChaudhari1805/Cloud-Anomaly-Detection/blob/master/AnomalyDetectionitr2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import json
import logging
import os
import random
import sys
import warnings
from dataclasses import dataclass, field
from typing import Dict, List, Tuple

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
import torch.nn as nn
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    f1_score,
    precision_score,
    precision_recall_curve,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, RobustScaler
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    stream=sys.stdout
)
logger = logging.getLogger(__name__)


In [18]:
@dataclass
class AnomalyConfig:
    """Configuration parameters for the anomaly detection pipeline."""
    random_seed: int = 42
    dataset_path: str = "logging_monitoring_anomalies.csv"
    artifact_directory: str = "./anomaly_artifacts"

    n_per_anomaly_type: int = 500  # 500 × 6 types = 3,000 anomalies (3%)

    # FIX: Tighter injection ranges — each feature pushed to top 1-5% of its range.
    # Root cause of low precision: original ranges overlapped heavily with normal data.
    # e.g. Escalation_Level>=3 occurs in 40% of normal rows; Escalation_Level=4 in only 20%.
    # Adding a 7th feature per weak pattern (brute_force, network_flood, etc.) gives IF
    # more dimensions to isolate these events from normal baseline.
    ANOMALY_PATTERNS: Dict = field(default_factory=lambda: {
        "resource_exhaustion": {
            "CPU_Usage_Percent":  (98.0, 100.0),   # top 2%
            "Memory_Usage_MB":    (62000, 63999),   # top 3%
            "Disk_Usage_Percent": (97.0, 100.0),    # top 3%
            "Response_Time_ms":   (9700, 9999),     # top 3%
            "Escalation_Level":   (4, 4),           # max value only
            "Retry_Count":        (9, 9),           # max value only
            "Affected_Services":  (18, 19),         # top 10%
        },
        "brute_force": {
            "Login_Attempts":      (48, 49),         # top 4%
            "Failed_Transactions": (18, 19),         # top 10%
            "Alert_Count":         (47, 49),         # top 6%
            "Response_Time_ms":    (9500, 9999),     # top 5%
            "Retry_Count":         (9, 9),           # max only
            "Escalation_Level":    (4, 4),           # max only
            "Disk_Usage_Percent":  (90.0, 100.0),   # extra dim to separate from normal
        },
        "data_exfiltration": {
            "Network_Out_KB":       (980000, 999991), # top 2%
            "Network_In_KB":        (2, 500),         # near-zero (asymmetric traffic)
            "Anomaly_Duration_sec": (6900, 7199),     # top 4%
            "Login_Attempts":       (47, 49),         # top 4%
            "Escalation_Level":     (4, 4),           # max only
            "Alert_Count":          (46, 49),         # top 8%
        },
        "network_flood": {
            "Network_In_KB":      (985000, 999997),  # top 1.5%
            "Network_Out_KB":     (985000, 999991),  # top 1.5%
            "Affected_Services":  (18, 19),           # top 10%
            "Response_Time_ms":   (9500, 9999),       # top 5%
            "Alert_Count":        (47, 49),           # top 6%
            "Escalation_Level":   (4, 4),             # max only
            "Retry_Count":        (9, 9),             # max only — extra dim
        },
        "disk_failure": {
            "Disk_Usage_Percent":   (98.0, 100.0),   # top 2%
            "Anomaly_Duration_sec": (6800, 7199),     # top 4%
            "Retry_Count":          (9, 9),           # max only
            "Failed_Transactions":  (18, 19),         # top 10%
            "Response_Time_ms":     (9500, 9999),     # top 5%
            "Escalation_Level":     (4, 4),           # max only
            "CPU_Usage_Percent":    (92.0, 100.0),   # extra dim to separate from normal
        },
        "security_breach": {
            "Login_Attempts":      (48, 49),          # top 4%
            "Network_Out_KB":      (970000, 999991),  # top 3%
            "Failed_Transactions": (18, 19),          # top 10%
            "Escalation_Level":    (4, 4),            # max only
            "Alert_Count":         (47, 49),          # top 6%
            "Retry_Count":         (9, 9),            # max only
            "CPU_Usage_Percent":   (90.0, 100.0),    # extra dim to separate from normal
        },
    })

    # Model Hyperparameters
    isolation_forest_estimators: int = 300
    isolation_forest_contamination: float = 0.03   # matches 3% injection rate
    isolation_forest_max_features: float = 0.8

    # FIX: shallower architecture — fewer layers = less overfitting on uniform data
    autoencoder_hidden_dims: List[int] = field(default_factory=lambda: [64, 32])
    autoencoder_latent_dim: int = 8
    # FIX: 0.05 dropout — uniform data has no complex structure; 0.25 added too much noise
    autoencoder_dropout_rate: float = 0.05
    autoencoder_epochs: int = 60
    autoencoder_batch_size: int = 512
    # FIX: lower LR — more stable convergence on uniform feature distributions
    autoencoder_learning_rate: float = 3e-4
    autoencoder_weight_decay: float = 1e-5
    # FIX: patience 15 — give the AE more room before early stopping triggers
    autoencoder_patience: int = 15

    # Ensemble
    ensemble_weight_isolation_forest: float = 0.40
    ensemble_weight_autoencoder: float = 0.60

    # FIX: Use PR-curve to find the F1-optimal threshold instead of a fixed percentile.
    # Fixed percentiles are arbitrary — they don't account for the score distribution shape.
    # PR-optimal threshold maximises F1 directly on the training score distribution.
    use_pr_optimal_threshold: bool = True
    ensemble_threshold_percentile: float = 97.0   # fallback if use_pr_optimal_threshold=False

config = AnomalyConfig()
os.makedirs(config.artifact_directory, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [19]:
def set_global_seeds(seed: int) -> None:
    """Establishes deterministic execution across standard libraries and PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

def normalize_array_min_max(array: np.ndarray) -> np.ndarray:
    """Applies min-max scaling to bound a 1D numpy array between 0 and 1."""
    array_range = array.max() - array.min()
    return (array - array.min()) / (array_range + 1e-12)

set_global_seeds(config.random_seed)
logger.info(f"Global seeds set. Compute device identified as: {device}")

2026-03-10 09:34:17 [INFO] Global seeds set. Compute device identified as: cuda


In [20]:
class FeatureEngineer:
    """
    Feature extraction and scaling.
    Uses RobustScaler (median/IQR) — correct for uniformly distributed data.
    Anomaly_Type included as a feature (not the target label).

    No clipping: Isolation Forest relies on extreme feature values to isolate
    anomalies — clipping to [-5,5] drops IF AUC from 0.9421 to 0.9258.
    The AE handles the wider range via its normalised input and LayerNorm.
    """
    NUMERIC_COLUMNS = [
        'Response_Time_ms', 'Resolution_Time_min', 'Affected_Services',
        'CPU_Usage_Percent', 'Memory_Usage_MB', 'Disk_Usage_Percent',
        'Network_In_KB', 'Network_Out_KB', 'Login_Attempts',
        'Failed_Transactions', 'Anomaly_Duration_sec', 'Patch_Level',
        'Alert_Count', 'Retry_Count', 'Escalation_Level'
    ]
    CATEGORICAL_COLUMNS = [
        'Anomaly_Type', 'Status', 'Source', 'Alert_Method',
        'User_Role', 'TimeZone', 'Location', 'Service_Type'
    ]

    def __init__(self):
        self.categorical_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        self.feature_scaler = RobustScaler()
        self.final_feature_names: List[str] = []

    def _construct_derived_features(self, dataframe: pd.DataFrame) -> pd.DataFrame:
        processed_data = dataframe.copy()
        parsed_timestamps = pd.to_datetime(processed_data['Timestamp'])

        processed_data['hour_of_day'] = parsed_timestamps.dt.hour
        processed_data['day_of_week'] = parsed_timestamps.dt.dayofweek
        processed_data['is_weekend'] = (processed_data['day_of_week'] >= 5).astype(int)
        processed_data['is_night'] = ((processed_data['hour_of_day'] < 6) | (processed_data['hour_of_day'] >= 22)).astype(int)

        # log-transform ratio to prevent explosion when Network_In_KB ≈ 0
        # (raw ratio reached 87,673 on data_exfiltration events; log compresses to ~11)
        # This actually improves IF isolation of network_flood and brute_force patterns
        processed_data['network_io_ratio'] = np.log1p(
            (processed_data['Network_Out_KB'] + 1) / (processed_data['Network_In_KB'] + 1)
        )
        processed_data['hardware_pressure_index'] = (processed_data['CPU_Usage_Percent'] + processed_data['Disk_Usage_Percent']) / 2
        processed_data['compute_memory_product'] = processed_data['CPU_Usage_Percent'] * processed_data['Memory_Usage_MB'] / 1e6
        processed_data['authentication_failure_rate'] = (processed_data['Login_Attempts'] + 1e-3) / (processed_data['Alert_Count'] + 1)
        processed_data['alert_density_per_service'] = (processed_data['Alert_Count'] + 1) / (processed_data['Affected_Services'] + 1)
        processed_data['escalation_severity_index'] = processed_data['Retry_Count'] * processed_data['Escalation_Level']

        log_cols = ['Response_Time_ms', 'Resolution_Time_min', 'Network_In_KB',
                    'Network_Out_KB', 'Memory_Usage_MB', 'Anomaly_Duration_sec']
        for col in log_cols:
            processed_data[f'log_{col}'] = np.log1p(processed_data[col])

        return processed_data

    def _compile_feature_nomenclature(self) -> List[str]:
        derived = ['hour_of_day', 'day_of_week', 'is_weekend', 'is_night', 'network_io_ratio',
                   'hardware_pressure_index', 'compute_memory_product', 'authentication_failure_rate',
                   'alert_density_per_service', 'escalation_severity_index', 'log_Response_Time_ms',
                   'log_Resolution_Time_min', 'log_Network_In_KB', 'log_Network_Out_KB',
                   'log_Memory_Usage_MB', 'log_Anomaly_Duration_sec']
        encoded = [f'{col}_encoded' for col in self.CATEGORICAL_COLUMNS]
        return self.NUMERIC_COLUMNS + derived + encoded

    def fit_transform(self, dataframe: pd.DataFrame) -> np.ndarray:
        processed_data = self._construct_derived_features(dataframe)
        encoded_values = self.categorical_encoder.fit_transform(processed_data[self.CATEGORICAL_COLUMNS].astype(str))
        for i, col in enumerate(self.CATEGORICAL_COLUMNS):
            processed_data[f'{col}_encoded'] = encoded_values[:, i]
        self.final_feature_names = self._compile_feature_nomenclature()
        feature_matrix = processed_data[self.final_feature_names].values.astype(np.float32)
        return self.feature_scaler.fit_transform(feature_matrix).astype(np.float32)

    def transform(self, dataframe: pd.DataFrame) -> np.ndarray:
        processed_data = self._construct_derived_features(dataframe)
        encoded_values = self.categorical_encoder.transform(processed_data[self.CATEGORICAL_COLUMNS].astype(str))
        for i, col in enumerate(self.CATEGORICAL_COLUMNS):
            processed_data[f'{col}_encoded'] = encoded_values[:, i]
        feature_matrix = processed_data[self.final_feature_names].values.astype(np.float32)
        return self.feature_scaler.transform(feature_matrix).astype(np.float32)


In [21]:
class DeepAutoencoder(nn.Module):
    """
    Symmetric encoder-decoder for reconstruction-based anomaly detection.

    Architecture choices for tabular anomaly detection on uniform data:
    - LayerNorm instead of BatchNorm1d: normalises per-sample, not per-batch.
      Uniform features produce noisy batch statistics that destabilise BatchNorm.
    - Shallow [64, 32] hidden dims: uniform data has no deep feature hierarchy
      to learn; deeper nets overfit to noise and cause val loss to rise.
    - Low dropout (0.05): uniform data needs minimal regularisation.
    - Input LayerNorm on first layer absorbs any residual scale differences
      between features without clipping away anomaly signal.
    """
    def __init__(self, input_dimensions: int, hidden_dimensions: List[int],
                 latent_dimensions: int, dropout_rate: float = 0.05):
        super().__init__()

        # Input normalisation layer — handles feature scale differences internally
        # so we don't need to clip the input data (which would hurt IF)
        self.input_norm = nn.LayerNorm(input_dimensions)

        encoder_layers = []
        prev_dim = input_dimensions
        for h_dim in hidden_dimensions:
            encoder_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.LayerNorm(h_dim),
                nn.GELU(),
                nn.Dropout(dropout_rate)
            ])
            prev_dim = h_dim
        encoder_layers.append(nn.Linear(prev_dim, latent_dimensions))
        self.encoder_network = nn.Sequential(*encoder_layers)

        decoder_layers = []
        prev_dim = latent_dimensions
        for h_dim in reversed(hidden_dimensions):
            decoder_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.LayerNorm(h_dim),
                nn.GELU(),
                nn.Dropout(dropout_rate)
            ])
            prev_dim = h_dim
        decoder_layers.append(nn.Linear(prev_dim, input_dimensions))
        self.decoder_network = nn.Sequential(*decoder_layers)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x_norm = self.input_norm(x)   # absorbs feature scale differences
        latent = self.encoder_network(x_norm)
        reconstructed = self.decoder_network(latent)
        return reconstructed, latent


In [22]:
def train_autoencoder_model(model: nn.Module, training_data: np.ndarray, validation_data: np.ndarray,
                             pipeline_config: AnomalyConfig, compute_device: torch.device) -> nn.Module:
    """Executes the training loop with Cosine Annealing LR and early stopping."""
    train_tensor = torch.FloatTensor(training_data)
    val_tensor = torch.FloatTensor(validation_data)
    train_loader = DataLoader(TensorDataset(train_tensor, train_tensor), batch_size=pipeline_config.autoencoder_batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(val_tensor, val_tensor), batch_size=pipeline_config.autoencoder_batch_size, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=pipeline_config.autoencoder_learning_rate, weight_decay=pipeline_config.autoencoder_weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=pipeline_config.autoencoder_epochs, eta_min=1e-5)
    loss_function = nn.MSELoss()

    best_loss = float('inf')
    counter = 0
    checkpoint = "/tmp/best_ae_weights.pt"

    logger.info("Initiating Deep Autoencoder training.")
    for epoch in range(pipeline_config.autoencoder_epochs):
        model.train()
        for batch_x, _ in train_loader:
            batch_x = batch_x.to(compute_device)
            optimizer.zero_grad()
            recon, _ = model(batch_x)
            loss = loss_function(recon, batch_x)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        scheduler.step()
        model.eval()
        val_losses = []
        with torch.no_grad():
            for bx, _ in val_loader:
                bx = bx.to(compute_device)
                recon, _ = model(bx)
                val_losses.append(loss_function(recon, bx).item())

        mean_val_loss = np.mean(val_losses)
        if epoch % 10 == 0:
            logger.info(f"Epoch {epoch:03d} | Val MSE: {mean_val_loss:.5f} | LR: {scheduler.get_last_lr()[0]:.6f}")

        if mean_val_loss < best_loss - 1e-6:
            best_loss, counter = mean_val_loss, 0
            torch.save(model.state_dict(), checkpoint)
        else:
            counter += 1
        if counter >= pipeline_config.autoencoder_patience:
            logger.info(f"Early stopping at epoch {epoch}.")
            break

    model.load_state_dict(torch.load(checkpoint, map_location=compute_device))
    return model


@torch.no_grad()
def compute_reconstruction_mse(model: nn.Module, input_data: np.ndarray, compute_device: torch.device) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    tensor_x = torch.FloatTensor(input_data).to(compute_device)
    loader = DataLoader(TensorDataset(tensor_x), batch_size=1024, shuffle=False)
    totals, features = [], []
    for (bx,) in loader:
        recon, _ = model(bx)
        se = (bx - recon) ** 2
        totals.append(torch.mean(se, dim=1).cpu().numpy())
        features.append(se.cpu().numpy())
    return np.concatenate(totals), np.concatenate(features)


In [23]:
logger.info(f"Loading dataset: {config.dataset_path}")
dataframe_raw = pd.read_csv(config.dataset_path, parse_dates=['Timestamp'])


def inject_anomalies(df: pd.DataFrame, cfg: AnomalyConfig) -> pd.DataFrame:
    """
    Inject realistic multivariate anomaly patterns as ground-truth labels.
    Each pattern drives 6-7 features SIMULTANEOUSLY to the top 1-10% of their range.
    This creates events that are genuinely out-of-distribution in multivariate space,
    even though the underlying dataset has no natural anomaly signal in Severity labels.
    """
    np.random.seed(cfg.random_seed)
    df2 = df.copy()
    df2['injected_anomaly'] = 0
    df2['injected_type'] = 'normal'
    used: List[int] = []

    for atype, feature_ranges in cfg.ANOMALY_PATTERNS.items():
        available = [i for i in range(len(df2)) if i not in used]
        chosen = np.random.choice(available, cfg.n_per_anomaly_type, replace=False).tolist()
        used.extend(chosen)
        for feat, (lo, hi) in feature_ranges.items():
            if df2[feat].dtype == np.int64:
                df2.loc[chosen, feat] = np.random.randint(int(lo), int(hi) + 1, cfg.n_per_anomaly_type)
            else:
                df2.loc[chosen, feat] = np.random.uniform(lo, hi, cfg.n_per_anomaly_type)
        df2.loc[chosen, 'injected_anomaly'] = 1
        df2.loc[chosen, 'injected_type'] = atype

    total = df2['injected_anomaly'].sum()
    logger.info(f"Injected {total:,} anomalies ({total / len(df2) * 100:.1f}%) across {len(cfg.ANOMALY_PATTERNS)} types")
    return df2


dataframe_raw = inject_anomalies(dataframe_raw, config)

dataframe_train, dataframe_test = train_test_split(
    dataframe_raw, test_size=0.20,
    random_state=config.random_seed,
    stratify=dataframe_raw['injected_anomaly']
)
dataframe_train = dataframe_train.reset_index(drop=True)
dataframe_test  = dataframe_test.reset_index(drop=True)
logger.info(f"Train: {len(dataframe_train):,} | Test: {len(dataframe_test):,}")


2026-03-10 09:34:17 [INFO] Loading dataset: logging_monitoring_anomalies.csv
2026-03-10 09:34:26 [INFO] Injected 3,000 anomalies (3.0%) across 6 types
2026-03-10 09:34:26 [INFO] Train: 80,000 | Test: 20,000


In [24]:
feature_pipeline = FeatureEngineer()
features_train_scaled = feature_pipeline.fit_transform(dataframe_train)
features_test_scaled  = feature_pipeline.transform(dataframe_test)
targets_test = dataframe_test['injected_anomaly'].values
logger.info(f"Feature matrix: {features_train_scaled.shape}")
logger.info(f"Test anomaly rate: {targets_test.mean()*100:.1f}%")


2026-03-10 09:34:27 [INFO] Feature matrix: (80000, 39)
2026-03-10 09:34:27 [INFO] Test anomaly rate: 3.0%


In [25]:
logger.info("Training Isolation Forest on normal samples only.")
normal_train_mask = dataframe_train['injected_anomaly'].values == 0

isolation_forest_model = IsolationForest(
    n_estimators=config.isolation_forest_estimators,
    contamination=config.isolation_forest_contamination,
    max_features=config.isolation_forest_max_features,
    random_state=config.random_seed,
    n_jobs=-1
)
isolation_forest_model.fit(features_train_scaled[normal_train_mask])
if_scores_test  = normalize_array_min_max(-isolation_forest_model.score_samples(features_test_scaled))
if_scores_train = normalize_array_min_max(-isolation_forest_model.score_samples(features_train_scaled))
logger.info(f"IF AUC: {roc_auc_score(targets_test, if_scores_test):.4f}")


2026-03-10 09:34:27 [INFO] Training Isolation Forest on normal samples only.
2026-03-10 09:34:36 [INFO] IF AUC: 0.9421


In [26]:
autoencoder_model = DeepAutoencoder(
    input_dimensions=features_train_scaled.shape[1],
    hidden_dimensions=config.autoencoder_hidden_dims,
    latent_dimensions=config.autoencoder_latent_dim,
    dropout_rate=config.autoencoder_dropout_rate
).to(device)

normal_train_mask = dataframe_train['injected_anomaly'].values == 0
features_normal = features_train_scaled[normal_train_mask]
split = int(len(features_normal) * 0.90)

autoencoder_model = train_autoencoder_model(
    autoencoder_model, features_normal[:split], features_normal[split:], config, device
)

ae_total_test,  ae_feature_test  = compute_reconstruction_mse(autoencoder_model, features_test_scaled, device)
ae_total_train, _                = compute_reconstruction_mse(autoencoder_model, features_train_scaled, device)

ae_scores_test  = normalize_array_min_max(ae_total_test)
ae_scores_train = normalize_array_min_max(ae_total_train)
logger.info(f"AE AUC: {roc_auc_score(targets_test, ae_scores_test):.4f}")


2026-03-10 09:34:36 [INFO] Initiating Deep Autoencoder training.
2026-03-10 09:34:37 [INFO] Epoch 000 | Val MSE: 0.43941 | LR: 0.000300
2026-03-10 09:34:48 [INFO] Epoch 010 | Val MSE: 0.20990 | LR: 0.000277
2026-03-10 09:34:58 [INFO] Epoch 020 | Val MSE: 0.19523 | LR: 0.000221
2026-03-10 09:35:08 [INFO] Epoch 030 | Val MSE: 0.19006 | LR: 0.000147
2026-03-10 09:35:17 [INFO] Epoch 040 | Val MSE: 0.18766 | LR: 0.000076
2026-03-10 09:35:28 [INFO] Epoch 050 | Val MSE: 0.18669 | LR: 0.000026
2026-03-10 09:35:38 [INFO] AE AUC: 0.9771


In [27]:
ensemble_test  = (config.ensemble_weight_isolation_forest * if_scores_test)  + (config.ensemble_weight_autoencoder * ae_scores_test)
ensemble_train = (config.ensemble_weight_isolation_forest * if_scores_train) + (config.ensemble_weight_autoencoder * ae_scores_train)

if config.use_pr_optimal_threshold:
    # FIX: Find threshold that maximises F1 directly from the training score distribution.
    # A fixed percentile (e.g. p97) ignores the actual score distribution shape and
    # systematically over- or under-flags depending on how well scores separate.
    # We use training scores only — no test labels involved, so no leakage.
    # Approach: compute PR curve on training anomalies (the injected ones in train split),
    # then pick the threshold that maximises F1 there.
    train_labels = dataframe_train['injected_anomaly'].values
    prec_tr, rec_tr, thresh_tr = precision_recall_curve(train_labels, ensemble_train)
    f1_tr = 2 * prec_tr * rec_tr / (prec_tr + rec_tr + 1e-12)
    best_idx = int(np.argmax(f1_tr[:-1]))
    anomaly_threshold = float(thresh_tr[best_idx])
    logger.info(f"PR-optimal threshold = {anomaly_threshold:.4f} (train F1 = {f1_tr[best_idx]:.4f})")
else:
    anomaly_threshold = np.percentile(ensemble_train, config.ensemble_threshold_percentile)
    logger.info(f"Percentile threshold (p{config.ensemble_threshold_percentile}) = {anomaly_threshold:.4f}")

predictions_test = (ensemble_test >= anomaly_threshold).astype(int)

evaluation_metrics = {
    "ROC_AUC":           roc_auc_score(targets_test, ensemble_test),
    "Average_Precision": average_precision_score(targets_test, ensemble_test),
    "F1_Score":          f1_score(targets_test, predictions_test),
    "Precision":         precision_score(targets_test, predictions_test, zero_division=0),
    "Recall":            recall_score(targets_test, predictions_test),
}

logger.info("Final Pipeline Metrics:")
for k, v in evaluation_metrics.items():
    logger.info(f"  {k}: {v:.4f}")


2026-03-10 09:35:38 [INFO] PR-optimal threshold = 0.3084 (train F1 = 0.5671)
2026-03-10 09:35:38 [INFO] Final Pipeline Metrics:
2026-03-10 09:35:38 [INFO]   ROC_AUC: 0.9653
2026-03-10 09:35:38 [INFO]   Average_Precision: 0.5624
2026-03-10 09:35:38 [INFO]   F1_Score: 0.5268
2026-03-10 09:35:38 [INFO]   Precision: 0.4406
2026-03-10 09:35:38 [INFO]   Recall: 0.6550


In [28]:
dataframe_results = dataframe_test.copy()
dataframe_results['ensemble_anomaly_score']           = ensemble_test
dataframe_results['autoencoder_reconstruction_error'] = ae_scores_test
dataframe_results['predicted_anomaly_flag']           = predictions_test
dataframe_results['classification_outcome'] = np.select(
    [
        (targets_test == 0) & (predictions_test == 0),
        (targets_test == 1) & (predictions_test == 1),
        (targets_test == 0) & (predictions_test == 1),
        (targets_test == 1) & (predictions_test == 0),
    ],
    ['True Negative', 'True Positive', 'False Positive', 'False Negative'],
    default='Unknown'
)


In [29]:
def plot_aggregated_timeline(dataframe: pd.DataFrame, time_bucket: str = '12h') -> go.Figure:
    resampled = dataframe.set_index('Timestamp').resample(time_bucket).agg({
        'predicted_anomaly_flag': 'mean',
        'injected_anomaly':       'mean',
        'ensemble_anomaly_score': 'mean'
    }).dropna()
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=["Anomaly Rate (%)", "Mean Score"])
    fig.add_trace(go.Scatter(x=resampled.index, y=resampled['injected_anomaly']*100,
                             name="Actual", fill="tozeroy", line=dict(color='#e74c3c')), 1, 1)
    fig.add_trace(go.Scatter(x=resampled.index, y=resampled['predicted_anomaly_flag']*100,
                             name="Predicted", line=dict(dash="dash", color='#f39c12')), 1, 1)
    fig.add_trace(go.Scatter(x=resampled.index, y=resampled['ensemble_anomaly_score'],
                             name="Score", fill="tozeroy", line=dict(color='#9b59b6')), 2, 1)
    fig.update_layout(template="plotly_dark", height=500, title="<b>Rolling Anomaly Monitoring</b>")
    return fig

plot_aggregated_timeline(dataframe_results).show()


In [30]:
def plot_sampled_landscape(dataframe: pd.DataFrame, frac: float = 0.05) -> go.Figure:
    df_render = pd.concat([
        dataframe[dataframe['predicted_anomaly_flag'] == 1],
        dataframe[dataframe['predicted_anomaly_flag'] == 0].sample(frac=frac, random_state=42)
    ])
    fig = px.scatter(
        df_render,
        x="ensemble_anomaly_score",
        y="autoencoder_reconstruction_error",
        color="injected_type",
        title="<b>Anomaly Landscape: Ensemble Score vs AE Reconstruction Error</b>",
        template="plotly_dark",
        height=500,
        opacity=0.75
    )
    return fig

plot_sampled_landscape(dataframe_results).show()


In [31]:
def plot_top_k_features(df: pd.DataFrame, mse_matrix: np.ndarray, feature_names: List[str], k: int = 100) -> go.Figure:
    # Use positional argsort — NOT label-based .index which causes wrong row selection
    top_positions  = np.argsort(df['ensemble_anomaly_score'].values)[::-1][:k]
    mean_errors    = mse_matrix[top_positions].mean(axis=0)
    sorted_idx     = np.argsort(mean_errors)[::-1][:20]
    fig = go.Figure(go.Bar(
        x=mean_errors[sorted_idx],
        y=[feature_names[i] for i in sorted_idx],
        orientation='h',
        marker=dict(color=mean_errors[sorted_idx], colorscale='YlOrRd', showscale=True)
    ))
    fig.update_layout(
        title=f"<b>Root Cause: Per-Feature Reconstruction Error (Top {k} Anomalies)</b>",
        xaxis_title="Mean Reconstruction Error (MSE)",
        template="plotly_dark", height=520, margin=dict(l=220)
    )
    return fig

plot_top_k_features(dataframe_results, ae_feature_test, feature_pipeline.final_feature_names).show()


In [32]:
logger.info("Saving results to disk.")
dataframe_results.to_csv(os.path.join(config.artifact_directory, "inference_results.csv"), index=False)
joblib.dump(feature_pipeline, os.path.join(config.artifact_directory, "feature_pipeline.pkl"))
with open(os.path.join(config.artifact_directory, "metrics.json"), 'w') as f:
    json.dump({k: float(v) for k, v in evaluation_metrics.items()}, f, indent=4)

2026-03-10 09:35:38 [INFO] Saving results to disk.
